### Spotify Listening Behavior Analysis and Skip Prediction

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [3]:
# Load data
df = pd.read_csv(
    r"C:\Users\hp\OneDrive\Desktop\data science\PROJECTS\spotify_ml_project\spotify_history.csv",
    encoding='latin1',
    low_memory=False
)
df

,ÿspotify_track_uri,ts,platform,ms_played,track_name,artist_name,album_name,reason_start,reason_end,shuffle,skipped
0,2J3n32GeLmMjwuAzyhcSNe,2013-07-08 02:44:34,web player,3185,"Say It, Just Say It",The Mowgli's,Waiting For The Dawn,autoplay,clickrow,False,False
1,1oHxIPqJyvAYHy0PVrDU98,2013-07-08 02:45:37,web player,61865,Drinking from the Bottle (feat. Tinie Tempah),Calvin Harris,18 Months,clickrow,clickrow,False,False
2,487OPlneJNni3NWC8SYqhW,2013-07-08 02:50:24,web player,285386,Born To Die,Lana Del Rey,Born To Die - The Paradise Edition,clickrow,unknown,False,False
3,5IyblF777jLZj1vGHG2UD3,2013-07-08 02:52:40,web player,134022,Off To The Races,Lana Del Rey,Born To Die - The Paradise Edition,trackdone,clickrow,False,False
4,0GgAAB0ZMllFhbNc3mAodO,2013-07-08 03:17:52,web player,0,Half Mast,Empire Of The Sun,Walking On A Dream,clickrow,nextbtn,False,False
...,...,...,...,...,...,...,...,...,...,...,...
149857,4Fz1WWr5o0OrlIcZxcyZtK,2024-12-15 23:06:19,android,1247,On The Way Home,John Mayer,Paradise Valley,fwdbtn,fwdbtn,True,True
149858,0qHMhBZqYb99yhX9BHcIkV,2024-12-15 23:06:21,android,1515,Magical Mystery Tour - Remastered 2009,The Beatles,Magical Mystery Tour,fwdbtn,fwdbtn,True,True
149859,0HHdujGjOZChTrl8lJWEIq,2024-12-15 23:06:22,android,1283,"Stop This Train - Live at the Nokia Theatre, L...",John Mayer,Where the Light Is: John Mayer Live In Los Ang...,fwdbtn,fwdbtn,True,True
149860,7peh6LUcdNPcMdrSH4JPsM,2024-12-15 23:06:23,android,1306,I Don't Trust Myself (With Loving You),John Mayer,Continuum,fwdbtn,fwdbtn,True,True


In [4]:
df.isnull().sum()

ÿspotify_track_uri      0
ts                      0
platform                0
ms_played               0
track_name              0
artist_name             0
album_name              4
reason_start          147
reason_end            121
shuffle                 4
skipped                 4
dtype: int64

In [5]:
df.drop('ÿspotify_track_uri', axis=1, inplace=True)

In [6]:
df['album_name'] = df['album_name'].replace(['', ' ', 'None'], np.nan).fillna('Unknown Album')
df['reason_start'] = df['reason_start'].replace(['', ' ', 'None'], np.nan).fillna('Unknown')
df['reason_end'] = df['reason_end'].replace(['', ' ', 'None'], np.nan).fillna('Unknown')

In [7]:
df['shuffle'] = df['shuffle'].fillna(0).astype(int)

In [8]:
df['skipped'] = (
    df['skipped']
    .astype(str)
    .str.lower()
    .str.strip()
    .map({'true': 1, 'false': 0, '1': 1, '0': 0})
    .fillna(0)
    .astype(int)
)

In [9]:
df.isnull().sum()

ts              0
platform        0
ms_played       0
track_name      0
artist_name     0
album_name      0
reason_start    0
reason_end      0
shuffle         0
skipped         0
dtype: int64

### skipp prediction

In [10]:
df.dtypes

ts              object
platform        object
ms_played       object
track_name      object
artist_name     object
album_name      object
reason_start    object
reason_end      object
shuffle          int64
skipped          int64
dtype: object

In [11]:
#  FEATURE ENGINEERING
df['ts'] = pd.to_datetime(df['ts'], errors='coerce')

In [12]:
df['hour'] = df['ts'].dt.hour
df['day'] = df['ts'].dt.day_name()
df['month'] = df['ts'].dt.month
df['weekend'] = df['day'].isin(['Saturday', 'Sunday']).astype(int)

In [13]:
df.drop('ts', axis=1, inplace=True)

In [14]:
df['ms_played'] = pd.to_numeric(df['ms_played'], errors='coerce')

In [15]:
from sklearn.preprocessing import LabelEncoder

cat_cols = [
    'platform',
    'track_name',
    'artist_name',
    'album_name',
    'reason_start',
    'reason_end',
    'day'
]

for col in cat_cols:
    le = LabelEncoder()
    df.loc[:, col] = le.fit_transform(df[col].astype(str))

In [16]:
X = df.drop('skipped', axis=1)
y = df['skipped']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = RandomForestClassifier()
model.fit(X_train, y_train)

In [ ]:
pred = model.predict(X_test)
pred

In [ ]:
print("Accuracy:", accuracy_score(y_test, pred))

### recommend songs

In [86]:
top_artists = df['artist_name'].value_counts().head(5)
print(top_artists)

artist_name
3502    13621
3604     6878
1775     4855
466      3814
2860     2697
Name: count, dtype: int64


In [87]:
recommended = df[df['artist_name'].isin(top_artists.index)]

recommended_songs = recommended[['track_name','artist_name']].drop_duplicates()
recommended_songs.head(10)

,track_name,artist_name
22,8762,1775
23,159,1775
24,1022,1775
25,4462,1775
26,9919,1775
27,12771,1775
28,5531,1775
29,2605,1775
30,2606,1775
31,4011,1775


In [88]:
liked_songs = df[df['skipped'] == 0]

liked_songs[['track_name','artist_name']].drop_duplicates().head(10)

,track_name,artist_name
0,9925,3643
1,3077,587
2,1532,2072
3,8416,2072
4,4555,1112
5,5731,1625
6,12916,2317
7,4908,3032
8,4613,3168
9,8934,1889
